A python notebook that tackles estimation of information loss due to using flat NER (as compared to nested NER).

In [1]:
from dataset_processing import cwed4eta_process_json_file, convert_to_token_spans

In [24]:
data_nested = cwed4eta_process_json_file("DATA/INF_LOSS/project-55-at-2026-03-12-07-36-c06d1e1c.json", [4])
data_flat = cwed4eta_process_json_file("DATA/INF_LOSS/project-55-at-2026-03-12-07-36-c06d1e1c.json", [1])

In [28]:
from collections import defaultdict

# 1. Create a dictionary from the flat data for fast lookup by ID
flat_dict = {item['id']: item for item in data_flat}

# Initialize counters for Step 2
total_nested_entities = 0
total_flat_entities = 0

# Initialize counters for Step 5
nested_counts_by_type = defaultdict(int)
lost_counts_by_type = defaultdict(int)
lost_entities_examples =[]

# Initialize counters for New Step (Embedded Spans)
spans_with_embedded_different_type = 0
embedded_examples =[]

# Process only the ~50 sentences present in the nested sample
for nested_doc in data_nested:
    doc_id = nested_doc['id']
    
    # Make sure this document exists in the flat dataset
    if doc_id not in flat_dict:
        continue 
        
    flat_doc = flat_dict[doc_id]
    
    # --- STEP 2: Absolute Counts ---
    total_nested_entities += len(nested_doc['entities'])
    total_flat_entities += len(flat_doc['entities'])
    
    # Create sets of tuples for exact matching: (start, end, label, text)
    nested_set = {(ent['start'], ent['end'], ent['label'], ent['text']) for ent in nested_doc['entities']}
    flat_set = {(ent['start'], ent['end'], ent['label'], ent['text']) for ent in flat_doc['entities']}
    
    # Calculate Lost Entities (False Negatives): in nested, but missing in flat
    lost_set = nested_set - flat_set
    
    # --- STEP 5: Type-Specific Counts ---
    for start, end, label, text in nested_set:
        nested_counts_by_type[label] += 1
        
    for start, end, label, text in lost_set:
        lost_counts_by_type[label] += 1
        lost_entities_examples.append({'id': doc_id, 'text': text, 'label': label})

    # --- NEW STEP: Proportion of Spans containing embedded spans of a different type ---
    for entA in nested_set:
        # We want to check if entA contains any other entity (entB)
        contains_different_type = False
        
        for entB in nested_set:
            if entA == entB:
                continue # Skip comparing the entity to itself
                
            # Check if entA physically contains entB
            if entA[0] <= entB[0] and entA[1] >= entB[1]:
                # Check if the labels are different
                if entA[2] != entB[2]:
                    contains_different_type = True
                    
                    # Save the example so we can inspect it later
                    embedded_examples.append({
                        'id': doc_id,
                        'parent_text': entA[3],
                        'parent_label': entA[2],
                        'child_text': entB[3],
                        'child_label': entB[2]
                    })
                    break # We found at least one, no need to keep checking entB for this entA
                    
        if contains_different_type:
            spans_with_embedded_different_type += 1

# --- PRINTING THE RESULTS ---

print("="*65)
print("STEP 2: ABSOLUTE ENTITY LOSS (SPAN-LEVEL)")
print("="*65)
absolute_loss = total_nested_entities - total_flat_entities
loss_rate = (absolute_loss / total_nested_entities) * 100 if total_nested_entities > 0 else 0

print(f"Total Entities in Nested (Gold): {total_nested_entities}")
print(f"Total Entities in Flat:          {total_flat_entities}")
print(f"Absolute Entities Lost:          {absolute_loss}")
print(f"Overall Information Loss Rate:   {loss_rate:.2f}%\n")


print("="*65)
print("STEP 5: LOSS CATEGORIZED BY ENTITY TYPE")
print("="*65)
print(f"{'Entity Label':<25} | {'Nested Total':<12} | {'Lost Count':<10} | {'Loss %':<10}")
print("-" * 65)

for label in sorted(nested_counts_by_type.keys()):
    nested_total = nested_counts_by_type[label]
    lost_count = lost_counts_by_type.get(label, 0)
    type_loss_rate = (lost_count / nested_total) * 100 if nested_total > 0 else 0
    print(f"{label:<25} | {nested_total:<12} | {lost_count:<10} | {type_loss_rate:.1f}%")


print("\n" + "="*65)
print("NEW STEP: PROPORTION OF SPANS CONTAINING EMBEDDED (DIFF TYPE)")
print("="*65)
embedded_rate = (spans_with_embedded_different_type / total_nested_entities) * 100 if total_nested_entities > 0 else 0
print(f"Total Nested Spans:                        {total_nested_entities}")
print(f"Spans Embedding a Different Entity Type:   {spans_with_embedded_different_type}")
print(f"Proportion:                                {embedded_rate:.2f}%\n")

print("Examples of Parent spans containing different Child spans:")
print("-" * 65)
# Print the first 5 examples to understand the nesting behavior
for ex in embedded_examples:
    print(f"Doc ID: {ex['id']}")
    print(f"  Parent: [{ex['parent_label']}] '{ex['parent_text']}'")
    print(f"  Child:  [{ex['child_label']}] '{ex['child_text']}'\n")

STEP 2: ABSOLUTE ENTITY LOSS (SPAN-LEVEL)
Total Entities in Nested (Gold): 651
Total Entities in Flat:          610
Absolute Entities Lost:          41
Overall Information Loss Rate:   6.30%

STEP 5: LOSS CATEGORIZED BY ENTITY TYPE
Entity Label              | Nested Total | Lost Count | Loss %    
-----------------------------------------------------------------
Asset                     | 27           | 2          | 7.4%
Body Part                 | 10           | 4          | 40.0%
Body of Water             | 11           | 0          | 0.0%
Chemical                  | 32           | 2          | 6.2%
Disease                   | 7            | 0          | 0.0%
Ecosystem                 | 7            | 4          | 57.1%
Energy Source             | 17           | 0          | 0.0%
Field of Study            | 16           | 0          | 0.0%
Geographical Feature      | 10           | 0          | 0.0%
Intellectual Artefact     | 50           | 4          | 8.0%
Location               